# Warpmate, Phase 4: custom CUDA inference

This notebook:

1. Builds Warpmate and runs the tests (the inference test checks every CUDA kernel against the CPU reference on made-up weights).
2. Gets the trained network: from `warpmate_model.zip` if you upload it (from the Phase 3 notebook), otherwise it trains it again (about 2 minutes).
3. Rebuilds the dataset (same seed as Phase 3, so the same test set).
4. Makes the **8-bit (INT8)** version of the network.
5. **Checks correctness**: every version against PyTorch on all 3.09 million test positions.
6. **Benchmarks** our kernels against PyTorch, `torch.compile` and TensorRT, and profiles them.

About 15 to 25 minutes. **Before you start:** *Runtime > Change runtime type > T4 GPU > Save*, then run the cells top to bottom.

In [ ]:
!nvidia-smi

## Get the code

If `warpmate-colab.zip` is in the **Files** panel on the left (drag it there), it's used; otherwise the code is cloned from GitHub.

In [ ]:
import os
os.chdir('/content')
if os.path.exists('/content/warpmate-colab.zip'):
    !mkdir -p Warpmate && unzip -o -q warpmate-colab.zip -d Warpmate
    print('Using the uploaded zip')
elif os.path.isdir('/content/Warpmate/.git'):
    !cd /content/Warpmate && git pull -q
    print('Updated the existing copy from GitHub')
else:
    !rm -rf Warpmate && git clone -q https://github.com/vasishtk1/Warpmate.git
    print('Cloned from GitHub')
%cd /content/Warpmate
!test -f src/inference/infer_gpu.cu && echo "Phase 4 code present" || echo "WARNING: this copy doesn't have the Phase 4 code yet"

## Build

Look for `Warpmate GPU support: ON`. This also builds `libwarpmate_infer.so`, the library Python calls.

In [ ]:
import os
if os.path.exists('/usr/local/cuda/bin/nvcc'):
    os.environ['CUDACXX'] = '/usr/local/cuda/bin/nvcc'
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -4
!cmake --build build -j 2>&1 | tail -4
!ls -la build/libwarpmate_infer.so

## Tests

`inference` should print one line per GPU variant (`naive`, `fused`, `sparse`, `int8`) with tiny differences from the CPU reference.

In [ ]:
!ctest --test-dir build --output-on-failure
!./build/warpmate-test-inference

## The trained network

To reuse the Phase 3 network, drag `warpmate_model.zip` into the **Files** panel before running this cell. Without it, the network is trained again below (the result will be very close but not identical).

In [ ]:
import os
if os.path.exists('/content/warpmate_model.zip') and not os.path.exists('model/warpmate_net.pt'):
    !mkdir -p model && unzip -o -q /content/warpmate_model.zip -d model
print('Network found' if os.path.exists('model/warpmate_net.pt') else 'No network yet: it will be trained after the dataset is built')

## Solve the tables and build the dataset

Same commands and seed as Phase 3, so the test set is the same 3,093,133 positions. (The Syzygy check was done in Phase 3 and is skipped here.)

In [ ]:
!pip -q install chess
!test -f tables/KQvKR.wmtb || ./build/warpmate-gen --gpu --all3 --all4 2>&1 | tail -2
!test -f data/dataset.npz || python ml/dataset.py --tables tables --out data/dataset.npz --per-table 1000000 | tail -4
!test -f model/warpmate_net.pt || python ml/train.py --data data/dataset.npz --out model --epochs 10

## Make the 8-bit network

Tries a few scales for the hidden layer on training positions (never the test set) and keeps the one that agrees most with the 32-bit network.

In [ ]:
!python ml/quantize.py --model model --data data/dataset.npz

## Correctness: every version against PyTorch

All 3.09 million test positions. The 32-bit versions (`cpu`, `naive`, `fused`, `sparse`) must match PyTorch to within 0.0001 in every probability and pick the same answer on every board that isn't a near-tie. The 8-bit versions must match the NumPy version of the 8-bit math and lose less than 1 point of accuracy. Look for `Overall: PASS`.

In [ ]:
!python ml/check_inference.py --data data/dataset.npz --model model --out model/inference_check.json

## Install TensorRT

NVIDIA's inference engine, the strongest thing to compare against. Takes a minute or two. If it fails, the benchmark still runs without it.

In [ ]:
!pip -q install tensorrt onnx

## Benchmark

Batch sizes 1 (one position), 32 (about one position's legal moves, what the dashboard needs), 256, 4,096 and 65,536 (bulk). Times are in microseconds per batch.

- **eager**: called from Python in a loop (includes Python and launch overhead; what the dashboard actually pays)
- **graph**: replayed as a CUDA graph (close to pure GPU time)

Building the TensorRT engines and compiling with `torch.compile` takes a few minutes. Ends with a per-kernel profile at batch 32 and 65,536.

In [ ]:
!python ml/bench_inference.py --data data/dataset.npz --model model --profile --out model/inference_bench.json

## Optional: download the results

The check and benchmark results (JSON) plus the 8-bit weights.

In [ ]:
!cd model && zip -q ../warpmate_phase4_results.zip inference_check.json inference_bench.json warpmate_net_int8.npz && cd .. && ls -la warpmate_phase4_results.zip
from google.colab import files
files.download('warpmate_phase4_results.zip')

## Done

Copy the output of every cell from **Tests** onward and paste it back (or download the notebook).